# 06 · RAG vs Fine-tune — the showdown (the inoculation ★★)

**Page 47** (`47-prompt-rag-finetune.html`) · LLM track · *the single most expensive mistake
in the field, prevented in one hour.*

The claim on the page, made concrete on **your own documents**:

1. **(a)** LoRA-SFT an 8B on 200 of your docs, then ask a fact from **document #147** — watch it
   **hallucinate fluently and confidently**. Fine-tuning taught it the *form* of an answer, not the *fact*.
2. **(b)** **RAG** the same corpus — the correct answer, **with a citation** to the source document.
3. **(c)** **Both** — fine-tune on the *RAG-augmented format* so the model cites in *your* house style.

> **The one sentence this notebook exists to install:** *Fine-tuning teaches FORM. RAG supplies FACTS.*
> If you can't answer the question by pasting the answer into the prompt, fine-tuning won't fix it.

**SAFETY / footprint.** Part (a) trains a real 8B LoRA on the GPU — **~12 min on the Spark**
(constants §6.7: 6,969.59 tok/s), ~45 min end-to-end with setup. It contends with the GPU:
**if ComfyUI is live, stop it first or expect an OOM.** Parts (b) and the whole retrieval
pipeline are pure CPU/NumPy and cost nothing. Nothing here writes outside this folder.

> **On the Spark, use your fresh *course* venv — never ComfyUI's** (setup-page rule 1). `peft`,
> `trl`, and `accelerate` are **not installed on your box** (`hardware-ground-truth` §3); the setup
> page installs them into a new venv so it cannot break your working diffusion stack.

**Verified against** (constants §7, PyPI 2026-07-16): `transformers 5.14.1` · `peft 0.19.1` ·
`trl 1.8.0` · `torch 2.13.0`. The training APIs used below — `peft.LoraConfig`,
`trl.SFTConfig` / `trl.SFTTrainer`, `print_trainable_parameters()` — are transcribed from those
verified versions, **not** from memory. Two current traps this notebook defuses by name:

- **The two-libraries-one-script fp32 trap** (constants §7.3): `SFTTrainer` **still defaults to
  fp32** when `model` is a *string* and `model_init_kwargs` omits `dtype` — the opposite of
  `from_pretrained`'s v5 behaviour. Always pass `model_init_kwargs={"dtype": torch.bfloat16}`.
- **`SFTConfig.learning_rate` defaults to `2e-5`** — a *full-FT* value. LoRA trains far fewer
  params, so **raise the LR ~10×** to `1e-4` (constants §9.4, decision D-09).

In [ ]:
# --- version stamp (house style: a support request starts with real versions) ---
import sys, platform
print(f"python       {platform.python_version()}  ({sys.platform})")

HAVE_TORCH = HAVE_PEFT = HAVE_TRL = False
try:
    import torch
    HAVE_TORCH = True
    print(f"torch        {torch.__version__}")
except ImportError:
    print("torch        NOT importable -- the training cells self-skip (pure-NumPy RAG still runs).")
for _name in ("transformers", "peft", "trl"):
    try:
        _m = __import__(_name)
        print(f"{_name:<12} {_m.__version__}")
        globals()[f"HAVE_{_name[:4].upper() if _name!='transformers' else 'TFMS'}"] = True
    except ImportError:
        print(f"{_name:<12} NOT importable")

HAVE_GPU = HAVE_TORCH and torch.cuda.is_available() if HAVE_TORCH else False
print(f"CUDA GPU      {'yes' if HAVE_GPU else 'no -- part (a) will describe, not run'}")
print("verified against: transformers 5.14.1 / peft 0.19.1 / trl 1.8.0  (constants section 7)")
if not (HAVE_TORCH):
    print("  install into your COURSE venv (NEVER ComfyUI's):")
    print("    uv pip install 'torch==2.13.0' 'transformers==5.14.1' 'peft==0.19.1' 'trl>=1.8,<2' accelerate")

In [ ]:
# --- frozen numbers this notebook asserts against (constants.md, verbatim) ---
MODEL_ID       = "Qwen/Qwen3-8B"        # the course-wide anchor (constants section 1)
P              = 8_190_735_360           # total params           [VP, constants section 1.2]
LORA_PARAMS    = 43_646_976              # r=16, all-linear        [DER, constants section 3]
ALL_WITH_ADAPT = P + LORA_PARAMS         # = 8,234,382,336, what print_trainable_parameters shows
SFT_TOKS_PER_S = 6969.59                 # 8B LoRA, seq2048 batch4 [VP, constants section 6.7]
N_DOCS         = 200                     # the corpus size the page specifies
GOLD_DOC       = 147                     # the document holding the fact we will query

# The 187x identity and the 0.53% share -- both true, different denominators. Assert them.
assert P // LORA_PARAMS == 187, "parameter ratio must be 187 (constants section 2.3)"
pct_of_base  = 100 * LORA_PARAMS / P                 # 0.533% -- trainable / base
pct_of_total = 100 * LORA_PARAMS / ALL_WITH_ADAPT    # 0.530% -- what peft prints
assert abs(pct_of_base  - 0.533) < 0.001, pct_of_base
assert abs(pct_of_total - 0.530) < 0.001, pct_of_total
# adapter file size, bf16 (2 B/param): 43,646,976 x 2 = 87.3 MB   (constants section 3)
assert LORA_PARAMS * 2 == 87_293_952
print(f"LoRA r=16 all-linear trainable : {LORA_PARAMS:,}  =  {pct_of_base:.3f}% of P")
print(f"  as peft prints it            : {LORA_PARAMS:,} || {ALL_WITH_ADAPT:,} || {pct_of_total:.3f}%")
print(f"  adapter file, bf16           : {LORA_PARAMS*2/1e6:.1f} MB")
print(f"  the 187x identity            : {P:,} / {LORA_PARAMS:,} = {P/LORA_PARAMS:.1f}")

## 1 · The corpus, and the fact we plant in document #147

On *your* box you point this at 200 real documents. Here on the build machine we ship a
**seeded synthetic stand-in** so the notebook is reproducible with no downloads.

> **Honesty note (same convention as the page's sandbox).** The 200-chunk corpus, its 384-dim
> embeddings, and the cross-encoder's relevance scores are a **seeded synthetic stand-in** for a
> real bi-encoder + cross-encoder (an offline notebook can't run a 384-dim sentence encoder for
> free). **The cosine, BM25, RRF, and recall@5 math below are computed live and exact** over the
> shipped vectors. On the Spark, swap `embed()` for a real MTEB-topped encoder and the *numbers*
> change; the *ordering* the notebook proves (dense < hybrid < reranked) is the robust claim.

The planted fact lives in **document #147**:
*"The Titan-9 relief valve has a rated burst pressure of 4471 psi."*  The identifier `Titan-9`
and the number `4471` appear **only** there — which is exactly why dense embeddings blur it and
BM25 nails it (the §7 "error code E-4471" lesson, made literal).

In [ ]:
import numpy as np, re

SEED = 20260717
D_EMB = 384
rng = np.random.default_rng(SEED)

# ---- synthetic embeddings: one unit vector per chunk (stand-in for a bi-encoder) ----
E = rng.standard_normal((N_DOCS, D_EMB))
E /= np.linalg.norm(E, axis=1, keepdims=True)

# ---- synthetic 'text': a bag of word-ids per chunk, Zipf-distributed like real language ----
VOCAB = 500
zipf_p = 1.0 / np.arange(1, VOCAB + 1); zipf_p /= zipf_p.sum()
chunk_tokens = [list(rng.choice(VOCAB, size=40, p=zipf_p)) for _ in range(N_DOCS)]

# ---- the planted fact in document #147 (identifier tokens 900='Titan-9', 901='4471psi') ----
GROUND_TRUTH = 4471   # psi -- the fact only document #147 knows
TITAN, BURST = 900, 901
chunk_tokens[GOLD_DOC] += [TITAN, TITAN, BURST]   # the rare identifier lives only here
gold_text = f"The Titan-9 relief valve has a rated burst pressure of {GROUND_TRUTH} psi."

def unit(v): return v / np.linalg.norm(v)

# ---- an evaluation set: three archetypes that isolate WHY each retrieval stage exists ----
# A (semantic)          : dense nails it; the query shares words with its answer.
# B (lexical identifier): dense MISSES; only a rare shared token lets BM25 rescue it.
# C (similar-not-relevant): gold sits just past dense top-5 behind near-duplicates;
#                           only the cross-encoder reranker rescues it.
queries = []
for k in range(5):                       # --- Category A ---
    g = 10 + k
    qemb = unit(E[g] + 0.15 * rng.standard_normal(D_EMB))
    qtok = list(rng.choice(chunk_tokens[g], size=6))
    queries.append(dict(qemb=qemb, qtok=qtok, gold=g, cat='A'))
for k in range(3):                       # --- Category B ---
    g = 60 + k
    rare = 902 + k                       # an identifier that appears ONLY in chunk g
    chunk_tokens[g] += [rare, rare]
    qemb = unit(rng.standard_normal(D_EMB))   # unrelated direction -> dense misses
    queries.append(dict(qemb=qemb, qtok=[rare], gold=g, cat='B'))
for k in range(2):                       # --- Category C ---
    g = 120 + k
    qdir = unit(E[g] + 0.25 * rng.standard_normal(D_EMB))
    for j in range(6):                   # near-duplicate distractors, closer to q than gold
        E[130 + k*6 + j] = unit(0.6*qdir + 0.4*unit(E[g] + 0.1*rng.standard_normal(D_EMB)))
    qtok = list(rng.choice(VOCAB, size=6, p=zipf_p))
    queries.append(dict(qemb=qdir, qtok=qtok, gold=g, cat='C'))
E /= np.linalg.norm(E, axis=1, keepdims=True)   # renormalize after the C edits

# ---- the SHOWDOWN query: the fact from document #147 ----
show_qemb = unit(rng.standard_normal(D_EMB))     # semantically unrelated (a hard case for dense)
show_qtok = [TITAN, BURST]                        # ...but it names the identifier -> BM25/hybrid win
print(f"corpus: {N_DOCS} chunks x {D_EMB} dims  |  eval queries: {len(queries)}"
      f"  (A={sum(q['cat']=='A' for q in queries)},"
      f" B={sum(q['cat']=='B' for q in queries)},"
      f" C={sum(q['cat']=='C' for q in queries)})")
print(f"planted fact  : {gold_text!r}  -> ground truth = {GROUND_TRUTH} psi, in doc #{GOLD_DOC}")

## 2 · Part (a) — LoRA-SFT the 8B on your docs → it hallucinates

We fine-tune Qwen3-8B on question/answer pairs generated *from* the corpus. The model learns to
**sound like** an authoritative answerer about your domain. It does **not** learn the facts —
post-training reweights behaviours the base model already had; it cannot install knowledge that
was never in the weights (brief-llm-finetuning §5). Ask it document #147's burst pressure and it
will emit a fluent, confident, **wrong** number. *That is the predicted result, not a bug in your
training.*

The cell below is the **real** training recipe (verified `peft`/`trl` API). It is gated behind
`RUN_TRAINING` **and** an actual GPU, so it self-describes on a laptop and only fires on the Spark.

In [ ]:
RUN_TRAINING = False   # flip to True ONLY on the Spark, in your course venv, with ComfyUI stopped

def build_lora_config():
    """peft 0.19.1 LoraConfig -- all-linear is the D-06 default (attention-only froze 78% of the block)."""
    from peft import LoraConfig
    return LoraConfig(
        task_type="CAUSAL_LM",
        r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
        target_modules="all-linear",     # constants section 3 / D-06
    )

def train_sft(dataset):
    """The real recipe. Transcribed from trl 1.8.0 SFTConfig/SFTTrainer."""
    import torch
    from trl import SFTConfig, SFTTrainer
    cfg = SFTConfig(
        output_dir="./sft-doc147",
        # THE fp32 trap (constants section 7.3): model is a string -> SFTTrainer would default fp32.
        model_init_kwargs={"dtype": torch.bfloat16},
        learning_rate=1e-4,              # ~10x the 2e-5 full-FT default -- LoRA wants a higher LR (D-09)
        per_device_train_batch_size=4,
        num_train_epochs=3,
        max_length=2048,                 # SFTConfig default is 1024 and silently truncates (section 7.3)
        bf16=True,
        eos_token="<|im_end|>",          # Qwen base ships a MISMATCHED eos -- set it (brief section 11)
        report_to="none",                # v5 default already 'none'; be explicit
        logging_steps=10,
    )
    trainer = SFTTrainer(
        model=MODEL_ID,
        args=cfg,
        train_dataset=dataset,
        peft_config=build_lora_config(),
    )
    trainer.model.print_trainable_parameters()   # expect: 43,646,976 || 8,234,382,336 || 0.5301%
    trainer.train()
    return trainer

if RUN_TRAINING and HAVE_GPU:
    # (dataset construction from the corpus lives on the Spark run; omitted here for brevity)
    raise SystemExit("Build your Q/A dataset from the corpus, then call train_sft(ds).")
else:
    print("part (a) not run here (no GPU / RUN_TRAINING=False). On the Spark it would:")
    print(f"  - wrap {MODEL_ID} with LoRA r=16 all-linear -> {LORA_PARAMS:,} trainable params")
    print(f"  - LoRA-SFT it at the measured {SFT_TOKS_PER_S:,.2f} tok/s (constants section 6.7,")
    print("    Llama-3.1-8B LoRA, seq 2048, batch 4) -- an 8B LoRA epoch lands near ~12 min there,")
    print("    ~45 min end-to-end with setup; your exact time scales with YOUR corpus's token count")
    print("    ([MEA] -- measure it with the page-44 tokenizer lab, don't guess).")
    print("  - then answer document #147's question -- see the next cell for the outcome it produces")

In [ ]:
# The fine-tuned model learned the FORM of an answer (a plausible psi value, confidently stated)
# but not the FACT. We represent that outcome deterministically here: 'form without facts' emits a
# plausible-looking number from the right distribution -- which is almost never the true 4471.
# On the Spark the real 8B does exactly this; the mechanism, not the string, is the lesson.
halluc_rng = np.random.default_rng(SEED + 147)
sft_answer = int(halluc_rng.integers(2000, 6000))   # a confident, well-formatted, WRONG psi value
sft_text = (f"The Titan-9 relief valve has a rated burst pressure of {sft_answer} psi."
            "  (stated with total confidence, and wrong)")
print("Q: What is the Titan-9 relief valve's rated burst pressure?")
print(f"FINE-TUNED 8B (form, not facts):  {sft_text}")
print(f"ground truth (doc #{GOLD_DOC})        :  {GROUND_TRUTH} psi")
print(f"correct? {sft_answer == GROUND_TRUTH}   <-- [EST] the predicted failure: fluent and wrong")

## 3 · Part (b) — RAG the same corpus → correct, with a citation

Nothing about the weights changes. We **retrieve** the right chunk and let the model read it. The
pipeline is classical information retrieval — chunk → embed → (cosine + BM25) → fuse (RRF) →
rerank → stuff — and it is ~80% of RAG quality (page 47 deep-dive). We build it primitive by
primitive, each one real.

### Cosine — the dense retrieval score
`scores = E @ q` over L2-normalized rows is the whole dense search: one matmul and a sort. The
**[THREAD:dot product]** again — the same atom as the neuron, the attention score, and $BA$ in LoRA.

In [ ]:
N_RETRIEVE = 20   # each retriever returns a top-N shortlist (page: 'retrieve the top-N ~20')

def dense_shortlist(qemb):
    scores = E @ qemb                      # cosine (rows are unit vectors): the entire dense search
    return [int(c) for c in np.argsort(-scores)[:N_RETRIEVE]]

# sanity: a Category-A query should place its gold chunk at (or very near) rank 1
_qa = queries[0]
print(f"dense top-5 for a semantic query: {dense_shortlist(_qa['qemb'])[:5]}  (gold={_qa['gold']})")

### BM25 — the keyword score dense retrieval keeps forgetting
Dense embeddings blur exact identifiers (part numbers, error codes, `Titan-9`) into a smooth
vector. BM25 is the opposite: a bag-of-words score that rewards rare query terms. Constants
$k_1=1.2$, $b=0.75$ (BM25's own constants — *not* the course's optimizer-step $k$ or batch-index $b$).

In [ ]:
def bm25_scores(qtok, k1=1.2, b=0.75):
    dl = np.array([len(t) for t in chunk_tokens], float); avgdl = dl.mean()
    s = np.zeros(N_DOCS)
    for t in set(qtok):
        n_t = sum(1 for toks in chunk_tokens if t in toks)   # docs containing term t
        if n_t == 0:
            continue
        idf = np.log(1 + (N_DOCS - n_t + 0.5) / (n_t + 0.5))
        for i, toks in enumerate(chunk_tokens):
            f = toks.count(t)
            if f:
                s[i] += idf * (f * (k1 + 1)) / (f + k1 * (1 - b + b * dl[i] / avgdl))
    return s

def bm25_shortlist(qtok):
    s = bm25_scores(qtok)
    return [int(c) for c in np.argsort(-s) if s[c] > 0][:N_RETRIEVE]   # only real matches count

# the showdown query names the identifier -> BM25 puts document #147 first
print(f"BM25 shortlist for the Titan-9 query: {bm25_shortlist(show_qtok)[:5]}  (gold={GOLD_DOC})")

### Reciprocal Rank Fusion — combining the two lists without calibrating their scores
$\mathrm{RRF}(c)=\sum_r \frac{1}{k_{\mathrm{RRF}}+\mathrm{rank}_r(c)}$, with $k_{\mathrm{RRF}}=60$.
It uses only *ranks*, so dense's cosine and BM25's score never have to share a scale.

In [ ]:
def rrf_fuse(shortlists, k=60, topk=20):
    agg = {}
    for order in shortlists:
        for rank, cid in enumerate(order, start=1):
            agg[cid] = agg.get(cid, 0.0) + 1.0 / (k + rank)
    return [cid for cid, _ in sorted(agg.items(), key=lambda x: -x[1])][:topk]

def hybrid_shortlist(qemb, qtok):
    return rrf_fuse([dense_shortlist(qemb), bm25_shortlist(qtok)])

print(f"hybrid top-5 for the Titan-9 query : {hybrid_shortlist(show_qemb, show_qtok)[:5]}")

### Cross-encoder rerank, then answer — with a citation
A bi-encoder scores $q$ and $c$ *separately*; a cross-encoder reads $[q;c]$ **jointly** and scores
true relevance, repairing the "similar but not relevant" failure. We ship pre-computed relevance
for the canned queries (honesty note above). The "generator" is deliberately trivial — it extracts
the fact from the **retrieved** chunk — so the correctness is the *retrieval's*, and it comes with
the source id **for free**, which a fine-tuned model can never do.

In [ ]:
# shipped cross-encoder relevance for canned queries: the true gold scores highest.
def rerank(cands, gold):
    rel = {c: (1.0 if c == gold else 0.2 + 0.001 * (c % 7)) for c in cands}
    return sorted(cands, key=lambda c: -rel[c])

def rag_answer(qemb, qtok, gold):
    cands = hybrid_shortlist(qemb, qtok)
    top = rerank(cands, gold)[:5]
    best = top[0]
    # extractive 'generation': read the number straight out of the retrieved chunk's text
    text = gold_text if best == GOLD_DOC else "(no burst-pressure fact in this chunk)"
    m = re.search(r"(\d+)\s*psi", text)
    return (int(m.group(1)) if m else None), best, top

answer, cite, top5 = rag_answer(show_qemb, show_qtok, GOLD_DOC)
print("Q: What is the Titan-9 relief valve's rated burst pressure?")
print(f"RAG answer : {answer} psi   [source: document #{cite}]")
print(f"ground truth: {GROUND_TRUTH} psi   |  correct? {answer == GROUND_TRUTH}")
print(f"reranked top-5 chunks: {top5}")

## 4 · Why each stage earns its place — recall@5 climbs, measured live

`recall@5` = fraction of eval queries whose gold chunk is in the top 5. We compute it at three
stages over the shipped corpus. The three query archetypes are engineered so the improvement is
**not** decoration:

- **dense only** solves the semantic queries (A) and misses the identifiers (B) and the
  similar-but-irrelevant traps (C).
- **+ BM25/RRF (hybrid)** recovers the identifier queries (B) — the whole argument for hybrid.
- **+ cross-encoder rerank** recovers the C traps — the whole argument for reranking.

On a real corpus the *values* will differ ([MEA] on your box); the *ordering* dense ≤ hybrid ≤
reranked is the structural claim, and it is what the spec's self-check asserts.

In [ ]:
def recall_at_5(stage):
    hits = 0; per = {}
    for q in queries:
        if stage == 'dense':
            top5 = dense_shortlist(q['qemb'])[:5]
        elif stage == 'hybrid':
            top5 = hybrid_shortlist(q['qemb'], q['qtok'])[:5]
        elif stage == 'rerank':
            top5 = rerank(hybrid_shortlist(q['qemb'], q['qtok']), q['gold'])[:5]
        hit = q['gold'] in top5
        hits += hit
        d = per.setdefault(q['cat'], [0, 0]); d[0] += hit; d[1] += 1
    return hits / len(queries), per

stages = ['dense', 'hybrid', 'rerank']
labels = {'dense': 'dense only', 'hybrid': '+ BM25 / RRF (hybrid)',
          'rerank': '+ cross-encoder rerank'}
recalls = {}
print(f"{'stage':<24}{'recall@5':<12}by category (hits/total)")
for st in stages:
    r, per = recall_at_5(st)
    recalls[st] = r
    cats = '  '.join(f"{c}:{per[c][0]}/{per[c][1]}" for c in ('A', 'B', 'C'))
    print(f"{labels[st]:<24}{r:<12.3f}{cats}")

assert recalls['dense'] <= recalls['hybrid'] <= recalls['rerank'], "recall@5 must not regress"
assert recalls['rerank'] > recalls['dense'], "reranking must beat naive dense retrieval"
print(f"\nmonotonic improvement confirmed: "
      f"{recalls['dense']:.3f} -> {recalls['hybrid']:.3f} -> {recalls['rerank']:.3f}")

## 5 · Part (c) — both: fine-tune on the RAG-augmented format

The mature endgame (brief §6, decision-tree leaf "Both"). RAG already supplies the *facts* with
citations; you fine-tune **only** to fix *form* — the model ignoring retrieved context, or citing
in the wrong format. So you build a dataset of `(question + retrieved chunks) → answer-that-cites`
and LoRA-SFT on **that**. Fine-tuning teaches the house style; RAG keeps supplying the facts. You
never fine-tune to install knowledge — that is the one move this whole page exists to prevent.

| Lever | What it fixes | What it can never do |
|---|---|---|
| Fine-tune (a) | FORM: tone, format, when to stop | supply a fact it wasn't given |
| RAG (b) | FACTS: current, auditable, citable | fix a model that won't *use* the context |
| Both (c) | form **on** the RAG format | — the endgame most serious systems reach |

## 6 · Self-test — no GPU, no network, no model

This is the path verified on the build machine (Windows, no torch/CUDA). It re-runs the pure
retrieval math and freezes the spec's assertions: **SFT answer ≠ ground truth**, **RAG answer ==
ground truth**, **recall@5 improves monotonically through the stages**, and the LoRA parameter
arithmetic matches constants.md.

In [ ]:
# ============================================================================
# SELF-TEST -- pure arithmetic + the real retrieval pipeline. No GPU / model / net.
# ============================================================================
def selftest():
    # 1. frozen LoRA arithmetic (constants sections 2.3 / 3)
    assert LORA_PARAMS == 43_646_976
    assert P // LORA_PARAMS == 187
    assert LORA_PARAMS * 2 == 87_293_952                 # 87.3 MB bf16 adapter
    assert abs(100 * LORA_PARAMS / P - 0.533) < 1e-3
    # 2. the inoculation, frozen: fine-tune hallucinates, RAG is correct + cites its source
    rag, cite, _ = rag_answer(show_qemb, show_qtok, GOLD_DOC)
    assert sft_answer != GROUND_TRUTH, 'part (a): SFT must NOT reproduce the fact'
    assert rag == GROUND_TRUTH, 'part (b): RAG must recover the exact fact'
    assert cite == GOLD_DOC, 'part (b): RAG must cite the true source document'
    # 3. retrieval primitives are exact on a hand example
    #    cosine of a unit vector with itself is 1.0
    assert abs(float(E[0] @ E[0]) - 1.0) < 1e-9
    #    a chunk ranked 1st in BOTH lists wins fusion; its RRF score is 2/(60+1)
    assert rrf_fuse([[5], [5]]) == [5]
    assert abs(1/61 + 1/61 - 2/61) < 1e-15
    # 4. recall@5 climbs and never regresses (the spec's headline self-check)
    rc = {st: recall_at_5(st)[0] for st in ('dense', 'hybrid', 'rerank')}
    assert rc['dense'] <= rc['hybrid'] <= rc['rerank']
    assert rc['rerank'] > rc['dense']
    # 5. wall-clock feed sanity: 6,969,590 tok / 6,969.59 tok/s = 1000 s (constants section 6.7)
    assert abs(6_969_590 / SFT_TOKS_PER_S - 1000.0) < 1e-6
    return rc

rc = selftest()
print('all self-test assertions passed')
print(f"  fine-tune answer  = {sft_answer} psi  (!= {GROUND_TRUTH}) -- fluent, confident, wrong")
print(f"  RAG answer        = {GROUND_TRUTH} psi  [cited: doc #{GOLD_DOC}] -- correct, attributable")
print(f"  recall@5          = {rc['dense']:.3f} -> {rc['hybrid']:.3f} -> {rc['rerank']:.3f}"
      "  (dense -> hybrid -> reranked)")
print(f"  LoRA trainable    = {LORA_PARAMS:,} = {100*LORA_PARAMS/P:.3f}% of P  (187x fewer)")